# LatentDriver Full Eval Dry-Run (Colab)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/achyutmorang/latentdriver-waymax-experiments/blob/main/notebooks/latentdriver_full_eval_colab.ipynb)

This notebook is the full-validation evaluation entry point. It assumes full preprocessing already completed and only orchestrates repository sync, Drive binding, artifact checks, dry-run validation, and optional full evaluation runs.


## Evaluation Contract

- Use the full WOMD validation split and the Drive-backed full preprocessing cache.
- Start with `full_reactive` and `latentdriver_t2_j3` dry-run only.
- Do not launch the full sweep until the dry-run and one-model run pass.
- Keep core logic in `scripts/*.py`; this notebook is an execution wrapper.


In [ ]:
from __future__ import annotations

import json
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/achyutmorang/latentdriver-waymax-experiments.git"
REPO_BRANCH = "main"
REPO_DIR = Path("/content/latentdriver-waymax-experiments")


def run(cmd: list[str], cwd: Path | None = None, env: dict[str, str] | None = None, stream: bool = True) -> str:
    printable = " ".join(str(part) for part in cmd)
    print(f"$ {printable}")
    if stream:
        proc = subprocess.Popen(
            cmd,
            cwd=str(cwd) if cwd else None,
            env=env,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1,
        )
        lines: list[str] = []
        assert proc.stdout is not None
        for line in proc.stdout:
            print(line, end="")
            lines.append(line)
        code = proc.wait()
        output = "".join(lines)
        if code != 0:
            raise RuntimeError(f"Command failed with exit code {code}: {printable}")
        return output
    completed = subprocess.run(
        cmd,
        cwd=str(cwd) if cwd else None,
        env=env,
        check=True,
        text=True,
        capture_output=True,
    )
    if completed.stdout:
        print(completed.stdout)
    if completed.stderr:
        print(completed.stderr, file=sys.stderr)
    return completed.stdout


if REPO_DIR.exists():
    run(["git", "fetch", "origin", REPO_BRANCH], cwd=REPO_DIR)
    run(["git", "checkout", REPO_BRANCH], cwd=REPO_DIR)
    run(["git", "pull", "--ff-only", "origin", REPO_BRANCH], cwd=REPO_DIR)
else:
    run(["git", "clone", "--branch", REPO_BRANCH, REPO_URL, str(REPO_DIR)])

print(run(["git", "rev-parse", "HEAD"], cwd=REPO_DIR, stream=False).strip())


In [ ]:
from __future__ import annotations

import json
import os
import sys

from google.colab import drive

drive.mount("/content/drive")

DRIVE_ROOT = "/content/drive/MyDrive/waymax_research"
sys.path.insert(0, str(REPO_DIR / "src"))

from latentdriver_waymax_experiments.colab import bind_drive_layout

binding = bind_drive_layout(DRIVE_ROOT)
print(json.dumps(binding, indent=2, sort_keys=True))


In [ ]:
from __future__ import annotations

import json
import os

WOMD_SOURCE_MODE = "gcs"  # Use "drive" only if you have all full validation shards staged locally.
WAYMO_GCS_ROOT = "gs://waymo_open_dataset_motion_v_1_1_0"
WAYMO_DATASET_DRIVE_ROOT = "/content/drive/MyDrive/waymax_research/latentdriver_waymax_experiments/assets/raw_womd/waymo_open_dataset_motion_v_1_1_0"

if WOMD_SOURCE_MODE == "gcs":
    os.environ["LATENTDRIVER_WAYMO_DATASET_ROOT"] = WAYMO_GCS_ROOT
elif WOMD_SOURCE_MODE == "drive":
    os.environ["LATENTDRIVER_WAYMO_DATASET_ROOT"] = WAYMO_DATASET_DRIVE_ROOT
else:
    raise ValueError(f"Unsupported WOMD_SOURCE_MODE={WOMD_SOURCE_MODE!r}")

print(json.dumps({
    "WOMD_SOURCE_MODE": WOMD_SOURCE_MODE,
    "LATENTDRIVER_WAYMO_DATASET_ROOT": os.environ["LATENTDRIVER_WAYMO_DATASET_ROOT"],
}, indent=2, sort_keys=True))


In [ ]:
from __future__ import annotations

import json

run([sys.executable, "scripts/bootstrap_upstream.py"], cwd=REPO_DIR)
lock_path = REPO_DIR / "artifacts" / "locks" / "upstream_bootstrap.json"
print(lock_path.read_text())


In [ ]:
INSTALL_RUNTIME = True

if INSTALL_RUNTIME:
    run([sys.executable, "scripts/setup_colab_runtime.py", "--editable-project"], cwd=REPO_DIR)
else:
    print("Skipping runtime installation. Set INSTALL_RUNTIME=True if this is a fresh Colab session.")


In [ ]:
EVAL_TIER = "full_reactive"  # Alternatives: "full_non_reactive"
EVAL_SEED = 0
DRY_RUN_MODEL = "latentdriver_t2_j3"
SINGLE_EVAL_MODEL = DRY_RUN_MODEL
MODELS = ["latentdriver_t2_j3", "latentdriver_t2_j4", "plant", "easychauffeur_ppo"]

RUN_SINGLE_EVAL = False  # Flip to True only after the dry-run is clean.
RUN_FULL_SWEEP = False   # Flip to True only after the one-model full eval succeeds.
RUN_PLOTS = False        # Flip to True after at least one full eval result exists.

if EVAL_TIER not in {"full_reactive", "full_non_reactive"}:
    raise ValueError(f"This notebook is for full eval tiers only, got {EVAL_TIER!r}")

print(json.dumps({
    "tier": EVAL_TIER,
    "seed": EVAL_SEED,
    "dry_run_model": DRY_RUN_MODEL,
    "single_eval_model": SINGLE_EVAL_MODEL,
    "models": MODELS,
    "run_single_eval": RUN_SINGLE_EVAL,
    "run_full_sweep": RUN_FULL_SWEEP,
    "run_plots": RUN_PLOTS,
}, indent=2, sort_keys=True))


In [ ]:
import json
import os
from pathlib import Path

FULL_PREPROCESSED_ROOT = REPO_DIR / "artifacts" / "assets" / "preprocessed" / "full"
PREPROCESS_ROOT = FULL_PREPROCESSED_ROOT / "val_preprocessed_path"
MAP_DIR = PREPROCESS_ROOT / "map"
ROUTE_DIR = PREPROCESS_ROOT / "route"
INTENTION_DIR = FULL_PREPROCESSED_ROOT / "val_intention_label"
SUCCESS_MARKER = PREPROCESS_ROOT / "_SUCCESS"
MANIFEST_PATH = PREPROCESS_ROOT / "preprocess_manifest.json"
EXPECTED_MIN_FULL_SCENARIOS = 44_000
CHECK_EXACT_ARTIFACT_COUNTS = False  # Leave False on Google Drive; scanning 44k files can raise FUSE I/O errors.


def safe_path_state(path: Path) -> dict[str, object]:
    try:
        return {
            "exists": path.exists(),
            "is_dir": path.is_dir(),
            "is_file": path.is_file(),
            "path": str(path),
        }
    except OSError as exc:
        return {
            "error": f"{type(exc).__name__}: {exc}",
            "path": str(path),
        }


def exact_count_files(path: Path, suffix: str) -> int:
    return sum(1 for entry in os.scandir(path) if entry.is_file() and entry.name.endswith(suffix))


path_status = {
    "map_dir": safe_path_state(MAP_DIR),
    "route_dir": safe_path_state(ROUTE_DIR),
    "intention_dir": safe_path_state(INTENTION_DIR),
    "success_marker": safe_path_state(SUCCESS_MARKER),
    "manifest": safe_path_state(MANIFEST_PATH),
}

if not SUCCESS_MARKER.is_file():
    raise RuntimeError(f"Missing preprocessing success marker: {SUCCESS_MARKER}")
if not MANIFEST_PATH.is_file():
    raise RuntimeError(f"Missing preprocessing manifest: {MANIFEST_PATH}")

manifest = json.loads(MANIFEST_PATH.read_text(encoding="utf-8"))
manifest_counts = manifest.get("counts", {})
if not isinstance(manifest_counts, dict):
    raise RuntimeError(f"Malformed preprocessing manifest counts: {manifest_counts!r}")

required_count_keys = ["map_npy", "route_npy", "intention_txt"]
missing_count_keys = [key for key in required_count_keys if key not in manifest_counts]
if missing_count_keys:
    raise RuntimeError(f"Preprocessing manifest is missing count keys: {missing_count_keys}")
if min(int(manifest_counts[key]) for key in required_count_keys) < EXPECTED_MIN_FULL_SCENARIOS:
    raise RuntimeError(f"Manifest counts are too low for full validation: {manifest_counts}")
if len({int(manifest_counts[key]) for key in required_count_keys}) != 1:
    raise RuntimeError(f"Manifest counts disagree across map/route/intention outputs: {manifest_counts}")

exact_counts = None
if CHECK_EXACT_ARTIFACT_COUNTS:
    exact_counts = {
        "map_npy": exact_count_files(MAP_DIR, ".npy"),
        "route_npy": exact_count_files(ROUTE_DIR, ".npy"),
        "intention_txt": exact_count_files(INTENTION_DIR, ".txt"),
    }
    if exact_counts != {key: int(manifest_counts[key]) for key in required_count_keys}:
        raise RuntimeError({"manifest_counts": manifest_counts, "exact_counts": exact_counts})

summary = {
    "full_preprocessed_root": str(FULL_PREPROCESSED_ROOT),
    "preprocess_root": str(PREPROCESS_ROOT),
    "path_status": path_status,
    "manifest_counts": manifest_counts,
    "exact_counts": exact_counts,
    "exact_count_scan_enabled": CHECK_EXACT_ARTIFACT_COUNTS,
}
print(json.dumps(summary, indent=2, sort_keys=True))


In [ ]:
import json

dry_run_output = run([
    sys.executable,
    "scripts/run_waymax_eval.py",
    "--model", DRY_RUN_MODEL,
    "--tier", EVAL_TIER,
    "--seed", str(EVAL_SEED),
    "--dry-run",
], cwd=REPO_DIR)

preflight = json.loads(dry_run_output)
print(json.dumps(preflight, indent=2, sort_keys=True))

if preflight.get("missing_inputs"):
    raise RuntimeError(
        "Full evaluation dry-run failed. Missing inputs detected.\n"
        + json.dumps(preflight["missing_inputs"], indent=2, sort_keys=True)
    )
if not preflight.get("ready"):
    raise RuntimeError("Full evaluation dry-run did not report ready=true.")


In [ ]:
import json
import pandas as pd

single_result = None

if RUN_SINGLE_EVAL:
    output = run([
        sys.executable,
        "scripts/run_waymax_eval.py",
        "--model", SINGLE_EVAL_MODEL,
        "--tier", EVAL_TIER,
        "--seed", str(EVAL_SEED),
    ], cwd=REPO_DIR)
    single_result = json.loads(output)
    summary = single_result.get("summary", {})
    display(pd.DataFrame([{
        "model": single_result["model"],
        "tier": single_result["tier"],
        "seed": single_result["seed"],
        "mAR[75:95]": summary.get("mar_75_95"),
        "AR[75:95]": summary.get("ar_75_95"),
        "offroad_rate": summary.get("offroad_rate"),
        "collision_rate": summary.get("collision_rate"),
        "progress_rate": summary.get("progress_rate"),
        "run_dir": single_result["run_dir"],
    }]))
else:
    print("Skipping full one-model eval. Set RUN_SINGLE_EVAL=True after the dry-run is clean.")


In [ ]:
import json
import pandas as pd

sweep_results = []

if RUN_FULL_SWEEP:
    for index, model in enumerate(MODELS, start=1):
        print(f"\n=== [{index}/{len(MODELS)}] {model} ===")
        output = run([
            sys.executable,
            "scripts/run_waymax_eval.py",
            "--model", model,
            "--tier", EVAL_TIER,
            "--seed", str(EVAL_SEED),
        ], cwd=REPO_DIR)
        sweep_results.append(json.loads(output))
else:
    print("Skipping full sweep. Set RUN_FULL_SWEEP=True only after the one-model full eval succeeds.")

rows = []
for item in sweep_results:
    summary = item.get("summary", {})
    rows.append({
        "model": item["model"],
        "tier": item["tier"],
        "seed": item["seed"],
        "mAR[75:95]": summary.get("mar_75_95"),
        "AR[75:95]": summary.get("ar_75_95"),
        "offroad_rate": summary.get("offroad_rate"),
        "collision_rate": summary.get("collision_rate"),
        "progress_rate": summary.get("progress_rate"),
        "run_dir": item["run_dir"],
    })

if rows:
    display(pd.DataFrame(rows))


In [ ]:
from pathlib import Path
from IPython.display import Image, display

if RUN_PLOTS:
    output = run([
        sys.executable,
        "scripts/plot_model_metrics.py",
        "--results-root", f"{DRIVE_ROOT}/latentdriver_waymax_experiments/results/runs",
        "--tier", EVAL_TIER,
        "--seed", str(EVAL_SEED),
    ], cwd=REPO_DIR)
    print(output)

    plots_root = Path(f"{DRIVE_ROOT}/latentdriver_waymax_experiments/results/runs/plots")
    candidates = sorted(plots_root.glob(f"*_{EVAL_TIER}_seed{EVAL_SEED}/model_metrics.png"))
    if candidates:
        latest_plot = candidates[-1]
        print(latest_plot)
        display(Image(filename=str(latest_plot)))
    else:
        print(f"No plot found under {plots_root} for tier={EVAL_TIER} seed={EVAL_SEED}")
else:
    print("Skipping plots. Set RUN_PLOTS=True after full eval metrics exist.")


## Next Step

If the dry-run passes, set `RUN_SINGLE_EVAL=True` and rerun the one-model eval cell. Only after that succeeds should you set `RUN_FULL_SWEEP=True` for all released checkpoints.
